In [1]:
import numpy as np

### 1. 交叉熵

$$ Loss = -\sum_{i=0}^{n} y_{true,i} \log(softmax(y_{pred,i})) $$

一般用于分类问题

In [2]:
def softmax(x):
    # 数值稳定
    x = x - np.max(x)

    return np.exp(x) / np.sum(np.exp(x), axis=-1, keepdims=True)

def cross_entropy(y_true, y_pred):
    y_pred = softmax(y_pred)

    return -np.sum((y_true * np.log(y_pred))) / y_true.shape[0]

### 2. MSE

均方误差损失

公式如下，

$$ Loss = \| y_{true} - y_{pred} \|_2^2 $$

一般用于回归问题

In [3]:
def MSE(y_true, y_pred):
    return np.sum((y_true - y_pred) ** 2)

### 3. NLLloss

英文全称：<span style="color: red">**N**</span>egative <span style="color: red">**L**</span>og-<span style="color: red">**L**</span>ikelihood <span style="color: red">**Loss**</span>

有如下等式成立：

$$ CrossEntropy = softmax + NLLloss $$

In [4]:
def NLLloss(y_true, y_pred):
    return -np.sum(y_true * np.log(y_pred)) / y_true.shape[0]

### 4. KL散度

有等式如下：

$$ D_{KL}(y_{true} || y_{pred}) = CrossEntropy(y_{true}, y_{pred}) - CrossEntropy(y_{true}, y_{true})$$

一般默认$ D_{KL} $中真实分布写在前面

也就是

$$ KL散度 = 交叉熵 - 信息熵 $$

公式如下：

$$ D_{KL} = -\sum_{i=0}^{n}y_{true,i} \cdot log\frac{y_{true,i}}{y_{pred,i}} $$

总结：

|名称|公式|
|:---:|:---:|
|交叉熵|$$-\sum_{i=0}^{n} y_{true,i} \log(softmax(y_{pred,i}))$$|
|信息熵|$$-\sum_{i=0}^{n} y_{true,i} \log(y_{true,i})$$|
|KL散度|$$\sum_{i=0}^{n}y_{true,i} \cdot log\frac{y_{true,i}}{softmax(y_{pred,i})}$$|

### KL永远为<span style="color: red">**非负数**</span>

原因：

由吉布斯不等式(不如直接用拉格朗日乘子法)计算得：

$$ -\sum_{i=0}^{n} y_{true,i} \log(softmax(y_{pred,i})) \geq  -\sum_{i=0}^{n} y_{true,i} \log(y_{true,i})$$

即：
$$ 交叉熵 \geq 信息熵 $$

因此：

$$ D_{KL} = CrossEntropy - Entropy \geq 0 $$

In [5]:
def kl_divergence(y_true, y_pred):
    return np.sum(y_true * np.log(y_true / softmax(y_pred))) / y_true.shape[0]

### 5. Focal Loss

交叉熵损失的升级版

公式如下：

$$ FL(p_t) = - \alpha_t \cdot (1 - p_t)^\gamma \cdot log(p_t) $$

其中，$ \alpha_t $ 和 $ \gamma $ 是都是超参数， $ \alpha_t $ 对于每一个类别应该有不同的权重

优点：

- 解决极端类别不平衡：下调海量简单样本（如背景）的损失权重，防止其梯度掩盖少数目标类
- 自适应难样本挖掘：无需手动抽样，通过指数因子动态聚焦预测概率低、学习难度大的样本
- 低侵入性即插即用：纯损失函数层面的改进，无需修改网络结构，直接替代交叉熵

缺点：

- 对标签噪声极度敏感：标注错误的样本（错标数据）会被误判为“极难样本”并赋予超高权重，导致模型严重过拟合
- 超参数调优成本高：引入聚焦参数 $ \gamma $ 与平衡因子 $ \alpha $，缺乏通用最优值，需针对不同数据集单独调参
- 抑制预测概率置信度：压缩了简单样本的响应，导致模型最终输出的概率估计普遍偏低
- 场景局限性强：在样本分布均衡的数据集上使用无收益，甚至可能降低模型性能

|数据集情况|是否推荐使用Focal Loss|替代方案/建议|
|:---:|:---:|:---:|
|严重不平衡 + 数据质量极高|强烈推荐|设 $ \gamma $ 为2，$ \alpha $ 为0.25|
|严重不平衡 + 数据存在较多噪声/错标|慎用|用Label Smoothing或带噪声抑制的Loss|
|数据基本平衡|不推荐|直接用普通的CrossEntropy|

In [6]:
def focal_loss(y_true, y_pred, alpha, gamma=0.25):
    return -alpha * ((1 - y_pred) ** gamma) * np.log(y_pred) 